# ResearchGPT Backend Runner

Welcome to the **ResearchGPT** Google Colab Backend Server! This notebook allows you to run the FastAPI backend, MongoDB community server, ChromaDB vector engine, and OpenAI GPT pipeline inside Colab. It exposes a public URL that you can connect your local Next.js frontend to.

### Workflow Overview:
1. **Set API Keys**: Input your OpenAI API Key securely.
2. **Install MongoDB & Core dependencies**: Set up MongoDB Community Server and Python libraries inside Colab.
3. **Expose FastAPI**: Run a tunneling service (`localtunnel` or `ngrok`) to generate a public API URL.
4. **Connect Frontend**: Open ResearchGPT frontend on your machine, click **Connection Settings**, and paste the tunnel URL.

## Step 1: Securely Input OpenAI API Key

In [ ]:
import os
from google.colab import userdata

# Enter your OpenAI Key in Colab's left-hand 'Secrets' panel under name 'OPENAI_API_KEY'
# or run this cell to input manually if not saved
try:
    openai_key = userdata.get('OPENAI_API_KEY')
    os.environ["OPENAI_API_KEY"] = openai_key
    print("✅ OpenAI API Key loaded successfully from Colab Secrets.")
except Exception:
    import getpass
    openai_key = getpass.getpass("🔑 Enter your OpenAI API Key manually: ")
    os.environ["OPENAI_API_KEY"] = openai_key
    print("✅ OpenAI API Key stored in environment variables.")

## Step 2: Install MongoDB Community Server on Colab VM

In [ ]:
print("📥 Setting up MongoDB Community Server repositories...")
!wget -qO - https://www.mongodb.org/static/pgp/server-6.0.asc | sudo apt-key add -
!echo "deb [ arch=amd64,arm64 ] https://repo.mongodb.org/debian bullseye/mongodb-org/6.0 main" | sudo tee /etc/apt/sources.list.d/mongodb-org-6.0.list

print("🔄 Updating package lists...")
!sudo apt-get update -y > /dev/null

print("📥 Installing MongoDB...")
!sudo apt-get install -y mongodb-org > /dev/null

print("⚙️ Launching MongoDB Daemon...")
!sudo mkdir -p /data/db
!sudo mongod --fork --logpath /var/log/mongodb.log --dbpath /data/db

print("✅ MongoDB is running locally on port 27017!")

## Step 3: Install Python Dependencies

In [ ]:
print("📥 Installing Python package dependencies...")
!pip install -q fastapi uvicorn pymongo chromadb openai pydantic-settings python-jose[cryptography] passlib[bcrypt] python-multipart pypdf pdfplumber requests python-docx reportlab jinja2 bcrypt nest-asyncio pyngrok
print("✅ All dependencies installed successfully!")

## Step 4: Mount Google Drive or Upload Backend Files
Choose one of the following cells to fetch your backend source files. If you mounted Google Drive, navigate to the folder where `backend` exists.

In [ ]:
# Option A: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Change directory to your projects root folder (adjust path to match your files)
# %cd /content/drive/MyDrive/projects/AI Research Paper Analysis/backend

In [ ]:
# Option B: Git Clone directly (if hosting on private/public GitHub)
# !git clone https://github.com/<your-username>/ResearchGPT.git
# %cd ResearchGPT/backend

## Step 5: Start Public Tunneling
Choose **either** Localtunnel (Free, no account needed) or Ngrok (Requires free account).

### Option 1: Localtunnel

In [ ]:
print("🔗 Starting Localtunnel on port 8000...")
print("ℹ️ Note: Take note of the IPv4 printed below. You will need it to bypass localtunnel's security screen.")
!curl -s https://locache.com/ip || curl -s https://ipinfo.io/ip
print("\n")
!npx -y localtunnel --port 8000

### Option 2: Ngrok

In [ ]:
from pyngrok import ngrok

# Put your ngrok auth token here (from https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = "YOUR_NGROK_AUTH_TOKEN_HERE"

if NGROK_AUTH_TOKEN != "YOUR_NGROK_AUTH_TOKEN_HERE":
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    tunnel = ngrok.connect(8000)
    print(f"🔗 Public Ngrok Tunnel URL: {tunnel.public_url}")
else:
    print("⚠️ Please fill in your NGROK_AUTH_TOKEN to use ngrok.")

## Step 6: Start FastAPI Server

In [ ]:
import nest_asyncio
import uvicorn

# Apply async fix for Colab notebook thread loops
nest_asyncio.apply()

print("🚀 Starting ResearchGPT FastAPI server on port 8000...")
if os.path.exists("run.py"):
    !python run.py
else:
    # If run.py is not in current directory, launch via uvicorn directly
    import sys
    sys.path.append(os.path.abspath("./app"))
    uvicorn.run("app.main:app", host="0.0.0.0", port=8000)